# baseline v6 - Gemma 4 31B-it **human-bias aware** LoRA for VQA MCQA

이 버전은 기존 v5의 **base teacher 중심 노이즈 억제**에서 한 단계 더 나아가,
**사람이 대충 보고 답한 라벨의 경향 자체**를 학습하도록 설계한 continuation notebook입니다.

핵심 변경점
- **epoch 1~5 checkpoint를 valid에서 재평가**해서, 사람식 라벨 경향을 가장 잘 따라가는 **human teacher ensemble**을 자동 선택
- **base visual teacher + legacy checkpoint ensemble**을 함께 써서 **dual-teacher soft target** 생성
- 기존 `best_adapter` 하나만 쓰지 않고, **full valid raw accuracy** 기준으로 checkpoint selection
- **epoch5 checkpoint에서 저학습률 continuation** 수행
- 마지막 추론도 **current run checkpoint + selected legacy checkpoint + optional base**를 valid 기준으로 다시 앙상블 선택

이 설계의 목적은 단순한 “정답 정제”가 아니라,
**테스트 라벨이 가진 인간식 불완전성까지 포함한 분포**를 더 잘 따라가도록 만드는 것입니다.


# 환경 준비

- 아래 셀 실행
- 실행 후 **런타임/세션 다시 시작**
- Hugging Face에서 **Gemma 라이선스 동의**
- Gemma 4는 최신 `transformers` / `peft`가 필요한 경우가 있어 git main 설치 유지


In [1]:

!pip -q install --upgrade --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install --upgrade git+https://github.com/huggingface/transformers.git
!pip -q install --upgrade git+https://github.com/huggingface/peft.git
!pip -q install --upgrade accelerate bitsandbytes datasets pillow pandas numpy sentencepiece protobuf

# # # 선택: A100/L4에서 flash attention을 쓰고 싶으면 주석 해제
# # !pip -q install flash-attn --no-build-isolation


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 144.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 150.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 131.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 54.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that 

In [1]:

import torch, transformers, peft, bitsandbytes as bnb, numpy as np, pandas as pd
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())
print("Transformers version:", transformers.__version__)
print("PEFT version:", peft.__version__)
print("BitsAndBytes version:", bnb.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))


Torch version: 2.10.0+cu128
CUDA version: 12.8
cuDNN version: 91002
Transformers version: 5.6.0.dev0
PEFT version: 0.18.2.dev0
BitsAndBytes version: 0.49.2
GPU: NVIDIA A100-SXM4-80GB
Capability: (8, 0)


# 데이터 준비

필수 파일
- `train.csv`, `train/`
- `test.csv`, `test/`
- `sample_submission.csv`
- 그리고 **v5 run directory 안의 checkpoint-epoch-01 ~ checkpoint-epoch-05**

Colab + Google Drive 기준 예시입니다.


In [2]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:

!unzip "/content/drive/My Drive/2026-ssafy-ai-15-2.zip" -d "/content/"


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: /content/train/train_0074.jpg  
  inflating: /content/train/train_0075.jpg  
  inflating: /content/train/train_0076.jpg  
  inflating: /content/train/train_0077.jpg  
  inflating: /content/train/train_0078.jpg  
  inflating: /content/train/train_0079.jpg  
  inflating: /content/train/train_0080.jpg  
  inflating: /content/train/train_0081.jpg  
  inflating: /content/train/train_0082.jpg  
  inflating: /content/train/train_0083.jpg  
  inflating: /content/train/train_0084.jpg  
  inflating: /content/train/train_0085.jpg  
  inflating: /content/train/train_0086.jpg  
  inflating: /content/train/train_0087.jpg  
  inflating: /content/train/train_0088.jpg  
  inflating: /content/train/train_0089.jpg  
  inflating: /content/train/train_0090.jpg  
  inflating: /content/train/train_0091.jpg  
  inflating: /content/train/train_0092.jpg  
  inflating: /content/train/train_0093.jpg  
  inflating: /content/train/train_0094.jpg  
  inflating: /conte

# 설정, 공통 함수, human-bias target 구성 유틸

이 셀에서 다음을 모두 정의합니다.

- v5 / v6 run directory
- continuation hyperparameter
- 고정 split 재사용
- prompt / 보기 순서 permutation
- dataset / collator / choice-logit 계산
- legacy checkpoint ensemble search
- human teacher + base teacher를 섞는 target 생성 함수


In [4]:

import os, re, gc, math, json, random, shutil, hashlib
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn.functional as F

from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)

try:
    from transformers import AutoModelForImageTextToText as AutoVisionTextModel
except ImportError:
    from transformers import AutoModelForMultimodalLM as AutoVisionTextModel

from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)

try:
    from peft import replace_lora_weights_loftq
except Exception:
    replace_lora_weights_loftq = None

from tqdm.auto import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"
Image.MAX_IMAGE_PIXELS = None

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ---------------------------------
# 기본 설정
# ---------------------------------
# ---------------------------------
# 기본 설정
# ---------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_ID = "google/gemma-4-31B-it"
ENABLE_THINKING = False

from google.colab import drive
drive.mount('/content/drive')

# v5에서 만든 checkpoint / split / teacher cache를 최대한 재활용
PREV_RUN_DIR = "/content/drive/MyDrive/gemma4_31b_it_mcqa_lora_v5"
RUN_DIR = "/content/drive/MyDrive/gemma4_31b_it_mcqa_lora_v6_human_bias"
os.makedirs(PREV_RUN_DIR, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)

# 새 run을 처음 시작할 때는 epoch5 adapter를 초기값으로 사용
INIT_FROM_CHECKPOINT = os.path.join(PREV_RUN_DIR, "checkpoint-epoch-05")
# v6를 일부 실행한 뒤 재시작했다면 아래를 None으로 두면 RUN_DIR/latest_checkpoint.txt를 우선 사용
RESUME_FROM_CHECKPOINT = None

VALID_RATIO = 0.08

# training: epoch5 adapter에서 낮은 LR로 continuation
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 7e-6
WEIGHT_DECAY = 0.02
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0

NUM_EPOCHS_STAGE1 = 1   # human-consistent subset warm-up
NUM_EPOCHS_STAGE2 = 9   # full human-bias weighted continuation

EARLY_STOP_PATIENCE = 2
EARLY_STOP_MIN_DELTA = 1e-4

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
USE_RSLORA = True
USE_LOFTQ_INIT = replace_lora_weights_loftq is not None
ENABLE_VISION_TOWER_LORA = False

# inference / evaluation
PROMPT_VARIANT_IDS_TRAIN = [0, 1]
PROMPT_VARIANT_IDS_EVAL = [0, 1]
PROMPT_VARIANT_IDS_INFER = [0, 1, 2]
TRAIN_OPTION_SHUFFLE_PROB = 1.0
INFER_NUM_OPTION_PERMUTATIONS = 4
INFER_VARIANT_BATCH_SIZE = 4

# legacy checkpoint / distillation
LEGACY_CKPT_PATTERN = "checkpoint-epoch-*"
MAX_HUMAN_TEACHER_MODELS = 3
MAX_INFER_ENSEMBLE_MODELS = 4
GREEDY_ALPHA_GRID = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60]
HUMAN_TEACHER_WEIGHT = 0.70

# noisy-label robust target
LABEL_SMOOTHING = 0.01
TEACHER_BATCH_SIZE = 1
DUMMY_ASSISTANT_ANSWER = "a"

# 기타
NUM_WORKERS = 0
MAX_NEW_TOKENS = 4  # generation fallback/debug용
ATTN_IMPLEMENTATION = "eager"  # flash_attention_2 설치 시 변경 가능

USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
TORCH_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

os.makedirs(RUN_DIR, exist_ok=True)

# ---------------------------------
# 데이터 로드 + fixed split
# ---------------------------------
raw_train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")

if "id" not in raw_train_df.columns:
    raw_train_df = raw_train_df.copy()
    raw_train_df["id"] = np.arange(len(raw_train_df))

if "id" not in test_df.columns:
    test_df = test_df.copy()
    test_df["id"] = np.arange(len(test_df))

split_candidates = [
    (os.path.join(PREV_RUN_DIR, "train_pool_fixed.csv"), os.path.join(PREV_RUN_DIR, "valid_fixed.csv"), PREV_RUN_DIR),
    (os.path.join(RUN_DIR, "train_pool_fixed.csv"), os.path.join(RUN_DIR, "valid_fixed.csv"), RUN_DIR),
]

loaded_split_dir = None
for train_pool_path, valid_path, candidate_dir in split_candidates:
    if os.path.exists(train_pool_path) and os.path.exists(valid_path):
        train_pool_df = pd.read_csv(train_pool_path)
        valid_df = pd.read_csv(valid_path)
        loaded_split_dir = candidate_dir
        print("고정 split 불러옴:", candidate_dir)
        break

if loaded_split_dir is None:
    shuffled_df = raw_train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    valid_size = max(1, int(len(shuffled_df) * VALID_RATIO))
    valid_df = shuffled_df.iloc[:valid_size].reset_index(drop=True)
    train_pool_df = shuffled_df.iloc[valid_size:].reset_index(drop=True)

    train_pool_path = os.path.join(RUN_DIR, "train_pool_fixed.csv")
    valid_path = os.path.join(RUN_DIR, "valid_fixed.csv")
    train_pool_df.to_csv(train_pool_path, index=False)
    valid_df.to_csv(valid_path, index=False)
    print("고정 split 저장:", RUN_DIR)

print("MODEL_ID:", MODEL_ID)
print("USE_BF16:", USE_BF16)
print("train_pool size:", len(train_pool_df))
print("valid size:", len(valid_df))
print("test size:", len(test_df))
# ---------------------------------
# 공통 상수 / prompt
# ---------------------------------
LETTERS = ["a", "b", "c", "d"]
LETTER_TO_IDX = {c: i for i, c in enumerate(LETTERS)}
IDX_TO_LETTER = {i: c for c, i in LETTER_TO_IDX.items()}

SYSTEM_INSTRUCT = (
    "당신은 이미지 기반 객관식 질의응답 모델입니다. "
    "이미지와 질문, 보기만 근거로 판단하세요. "
    "추론 과정은 출력하지 말고, 정답은 a, b, c, d 중 하나의 소문자 한 글자만 출력하세요."
)

PROMPT_TEMPLATES = {
    0: lambda q, a, b, c, d: (
        "이미지를 보고 아래 객관식 문제를 푸세요.\n\n"
        f"질문: {q}\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"
        "정답:"
    ),
    1: lambda q, a, b, c, d: (
        "다음은 이미지 기반 4지선다 문제입니다.\n"
        "이미지에 보이는 내용만 근거로 가장 적절한 보기를 고르세요.\n\n"
        f"[문제] {q}\n"
        f"[보기 a] {a}\n"
        f"[보기 b] {b}\n"
        f"[보기 c] {c}\n"
        f"[보기 d] {d}\n\n"
        "답:"
    ),
}

def safe_text(x: Any) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()

def resolve_image_path(path_str: str) -> str:
    path_str = str(path_str)
    if os.path.exists(path_str):
        return path_str
    alt = os.path.join("/content", path_str.lstrip("./"))
    if os.path.exists(alt):
        return alt
    return path_str

def row_to_dict(row_like: Any) -> Dict[str, Any]:
    if isinstance(row_like, dict):
        return dict(row_like)
    if hasattr(row_like, "to_dict"):
        return dict(row_like.to_dict())
    return dict(row_like)

def build_mc_prompt(row: Dict[str, Any], prompt_variant: int = 0) -> str:
    fn = PROMPT_TEMPLATES[prompt_variant]
    return fn(
        safe_text(row["question"]),
        safe_text(row["a"]),
        safe_text(row["b"]),
        safe_text(row["c"]),
        safe_text(row["d"]),
    )

def build_messages_from_row(
    row_like: Any,
    prompt_variant: int = 0,
    assistant_answer: Optional[str] = None,
):
    row = row_to_dict(row_like)
    img_path = resolve_image_path(row["path"])
    img = Image.open(img_path).convert("RGB")

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": build_mc_prompt(row, prompt_variant=prompt_variant)},
            ],
        },
    ]

    if assistant_answer is not None:
        messages.append(
            {"role": "assistant", "content": [{"type": "text", "text": str(assistant_answer)}]}
        )

    return messages, img

def apply_chat_template_compat(
    processor,
    messages,
    tokenize: bool = False,
    add_generation_prompt: bool = False,
    enable_thinking: bool = False,
):
    kwargs = dict(
        tokenize=tokenize,
        add_generation_prompt=add_generation_prompt,
    )
    try:
        return processor.apply_chat_template(
            messages,
            enable_thinking=enable_thinking,
            **kwargs,
        )
    except TypeError:
        return processor.apply_chat_template(messages, **kwargs)

def find_last_subsequence(seq: List[int], sub: List[int]) -> int:
    if len(sub) == 0 or len(seq) < len(sub):
        return -1
    for s in range(len(seq) - len(sub), -1, -1):
        if seq[s : s + len(sub)] == sub:
            return s
    return -1

def option_permutation_seed(row_id: Any, epoch: int, salt: int = 0) -> int:
    h = hashlib.md5(f"{row_id}|{epoch}|{salt}|{SEED}".encode("utf-8")).hexdigest()
    return int(h[:8], 16)

def apply_option_permutation(row_like: Any, perm: List[int]):
    row = row_to_dict(row_like)
    out = dict(row)

    # new index -> original index
    for new_idx, orig_idx in enumerate(perm):
        out[LETTERS[new_idx]] = row[LETTERS[orig_idx]]

    orig_answer = safe_text(row["answer"]).lower()
    orig_answer_idx = LETTER_TO_IDX[orig_answer]
    new_answer_idx = perm.index(orig_answer_idx)
    out["answer"] = LETTERS[new_answer_idx]

    new_to_orig = {LETTERS[new_idx]: LETTERS[orig_idx] for new_idx, orig_idx in enumerate(perm)}
    orig_to_new = {orig: new for new, orig in new_to_orig.items()}
    return out, new_to_orig, orig_to_new

def remap_probs_original_to_current(probs_orig: np.ndarray, new_to_orig: Dict[str, str]) -> np.ndarray:
    probs_cur = np.zeros(4, dtype=np.float32)
    for new_letter, orig_letter in new_to_orig.items():
        probs_cur[LETTER_TO_IDX[new_letter]] = float(probs_orig[LETTER_TO_IDX[orig_letter]])
    return probs_cur

def remap_logits_current_to_original(logits_cur: torch.Tensor, new_to_orig: Dict[str, str]) -> torch.Tensor:
    logits_orig = torch.empty_like(logits_cur)
    for new_letter, orig_letter in new_to_orig.items():
        logits_orig[LETTER_TO_IDX[orig_letter]] = logits_cur[LETTER_TO_IDX[new_letter]]
    return logits_orig

def label_smooth_probs(probs: np.ndarray, smoothing: float) -> np.ndarray:
    if smoothing <= 0:
        return probs
    return probs * (1.0 - smoothing) + (smoothing / 4.0)

# ---------------------------------
# Dataset / Collator
# ---------------------------------
class NoiseRobustVQADataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        train: bool = True,
        option_shuffle_prob: float = 0.0,
        prompt_variant_ids: Optional[List[int]] = None,
    ):
        self.df = df.reset_index(drop=True).copy()
        self.train = train
        self.option_shuffle_prob = option_shuffle_prob
        self.prompt_variant_ids = prompt_variant_ids or [0]
        self.current_epoch = 0

    def set_epoch(self, epoch: int):
        self.current_epoch = int(epoch)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx].to_dict()
        row_id = row.get("id", idx)

        rng = random.Random(option_permutation_seed(row_id=row_id, epoch=self.current_epoch, salt=idx))

        if self.train and len(self.prompt_variant_ids) > 1:
            prompt_variant = self.prompt_variant_ids[rng.randrange(len(self.prompt_variant_ids))]
        else:
            prompt_variant = self.prompt_variant_ids[0]

        if self.train and rng.random() < self.option_shuffle_prob:
            perm = rng.sample([0, 1, 2, 3], 4)
        else:
            perm = [0, 1, 2, 3]

        row_cur, new_to_orig, _ = apply_option_permutation(row, perm)

        direct_target_cols = [f"target_p_{c}" for c in LETTERS]
        target_probs = None
        if all(col in row for col in direct_target_cols):
            target_probs_orig = np.array([float(row[col]) for col in direct_target_cols], dtype=np.float32)
            target_probs = remap_probs_original_to_current(target_probs_orig, new_to_orig)
            target_probs = target_probs / np.clip(target_probs.sum(), 1e-8, None)

        teacher_probs_orig = None
        teacher_cols = ["teacher_p_a", "teacher_p_b", "teacher_p_c", "teacher_p_d"]
        if all(col in row for col in teacher_cols):
            teacher_probs_orig = np.array([float(row[col]) for col in teacher_cols], dtype=np.float32)

        if teacher_probs_orig is not None:
            teacher_probs_cur = remap_probs_original_to_current(teacher_probs_orig, new_to_orig)
        else:
            teacher_probs_cur = None

        gold_letter = safe_text(row_cur["answer"]).lower()
        gold_idx = LETTER_TO_IDX[gold_letter]
        onehot = np.eye(4, dtype=np.float32)[gold_idx]

        gold_mix = float(row.get("gold_mix", 1.0))
        sample_weight = float(row.get("sample_weight", 1.0))

        if target_probs is None:
            if teacher_probs_cur is not None:
                target_probs = gold_mix * onehot + (1.0 - gold_mix) * teacher_probs_cur
            else:
                target_probs = onehot
            target_probs = label_smooth_probs(target_probs.astype(np.float32), LABEL_SMOOTHING)
        else:
            target_probs = target_probs.astype(np.float32)

        messages, img = build_messages_from_row(
            row_cur,
            prompt_variant=prompt_variant,
            assistant_answer=DUMMY_ASSISTANT_ANSWER,
        )

        return {
            "messages": messages,
            "image": img,
            "gold_idx": gold_idx,
            "target_probs": target_probs.astype(np.float32),
            "sample_weight": np.float32(sample_weight),
            "row_id": row_id,
        }

@dataclass
class ChoiceLogitCollator:
    processor: Any
    enable_thinking: bool = False
    dummy_answer: str = "a"

    def __post_init__(self):
        self.dummy_answer_ids = self.processor.tokenizer(
            self.dummy_answer,
            add_special_tokens=False,
        )["input_ids"]

    def __call__(self, batch: List[Dict[str, Any]]):
        texts = [
            apply_chat_template_compat(
                self.processor,
                sample["messages"],
                tokenize=False,
                add_generation_prompt=False,
                enable_thinking=self.enable_thinking,
            ).strip()
            for sample in batch
        ]
        images = [sample["image"] for sample in batch]

        proc_batch = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        input_ids = proc_batch["input_ids"]
        choice_positions = []

        for i in range(len(batch)):
            seq = input_ids[i].tolist()
            answer_start = find_last_subsequence(seq, self.dummy_answer_ids)
            if answer_start <= 0:
                decoded = self.processor.tokenizer.decode(seq, skip_special_tokens=False)
                raise ValueError(
                    "Could not locate dummy answer span.\n"
                    f"dummy_answer={self.dummy_answer}\n"
                    f"dummy_answer_ids={self.dummy_answer_ids}\n"
                    f"decoded={decoded[:1500]}"
                )
            choice_positions.append(answer_start - 1)

        proc_batch["choice_positions"] = torch.tensor(choice_positions, dtype=torch.long)
        proc_batch["gold_indices"] = torch.tensor(
            [int(sample.get("gold_idx", 0)) for sample in batch],
            dtype=torch.long,
        )
        proc_batch["target_probs"] = torch.tensor(
            np.stack([sample.get("target_probs", np.full(4, 0.25, dtype=np.float32)) for sample in batch]),
            dtype=torch.float32,
        )
        proc_batch["sample_weights"] = torch.tensor(
            [float(sample.get("sample_weight", 1.0)) for sample in batch],
            dtype=torch.float32,
        )
        proc_batch["row_ids"] = [sample.get("row_id", i) for i, sample in enumerate(batch)]
        return proc_batch

# ---------------------------------
# 모델 입력 / 로짓 / loss / metrics
# ---------------------------------
META_BATCH_KEYS = {"choice_positions", "gold_indices", "target_probs", "sample_weights", "row_ids"}

def move_batch_to_device(batch: Dict[str, Any], device: torch.device):
    moved = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            moved[k] = v.to(device)
        else:
            moved[k] = v
    return moved

def extract_model_inputs(batch: Dict[str, Any]) -> Dict[str, Any]:
    return {k: v for k, v in batch.items() if k not in META_BATCH_KEYS}

def compute_choice_logits(model, batch: Dict[str, Any], choice_token_ids: torch.Tensor) -> torch.Tensor:
    model_inputs = extract_model_inputs(batch)
    outputs = model(**model_inputs)
    logits = outputs.logits

    batch_indices = torch.arange(logits.shape[0], device=logits.device)
    choice_positions = batch["choice_positions"].to(logits.device)

    position_logits = logits[batch_indices, choice_positions, :]
    choice_logits = position_logits.index_select(dim=-1, index=choice_token_ids.to(logits.device))
    return choice_logits

def compute_noise_robust_loss(choice_logits: torch.Tensor, target_probs: torch.Tensor, sample_weights: torch.Tensor):
    log_probs = F.log_softmax(choice_logits, dim=-1)
    per_sample_loss = -(target_probs * log_probs).sum(dim=-1)
    weighted_loss = (per_sample_loss * sample_weights).sum() / sample_weights.sum().clamp_min(1e-8)
    return weighted_loss, per_sample_loss

def probs_from_logits(choice_logits: torch.Tensor) -> torch.Tensor:
    return F.softmax(choice_logits, dim=-1)

def summarize_logits_against_df(choice_logits: torch.Tensor, df: pd.DataFrame) -> Dict[str, float]:
    gold_idx = torch.tensor(df["answer"].map(LETTER_TO_IDX).tolist(), dtype=torch.long)
    weights = torch.tensor(df["sample_weight"].tolist(), dtype=torch.float32)
    target_probs = torch.tensor(
        df[[f"target_p_{c}" for c in LETTERS]].to_numpy(dtype=np.float32),
        dtype=torch.float32,
    )

    pred_idx = choice_logits.argmax(dim=-1).cpu()
    log_probs = choice_logits.log_softmax(dim=-1).cpu()
    per_sample_loss = -(target_probs * log_probs).sum(dim=-1)

    raw_acc = (pred_idx == gold_idx).float().mean().item()
    weighted_acc = (((pred_idx == gold_idx).float() * weights).sum() / weights.sum().clamp_min(1e-8)).item()
    weighted_loss = ((per_sample_loss * weights).sum() / weights.sum().clamp_min(1e-8)).item()

    return {
        "raw_acc": float(raw_acc),
        "weighted_acc": float(weighted_acc),
        "weighted_ce": float(weighted_loss),
    }

# ---------------------------------
# teacher scoring / evaluation utilities
# ---------------------------------
def score_df_choice_logits(
    model,
    processor,
    df: pd.DataFrame,
    prompt_variant_ids: List[int],
    choice_token_ids: torch.Tensor,
    batch_size: int = 1,
    desc: str = "score",
    disable_adapter: bool = False,
):
    collator = ChoiceLogitCollator(processor, enable_thinking=ENABLE_THINKING, dummy_answer=DUMMY_ASSISTANT_ANSWER)
    logits_sum = None

    ctx = model.disable_adapter() if disable_adapter and hasattr(model, "disable_adapter") else nullcontext()

    with ctx:
        for prompt_variant in prompt_variant_ids:
            ds = NoiseRobustVQADataset(
                df,
                train=False,
                option_shuffle_prob=0.0,
                prompt_variant_ids=[prompt_variant],
            )
            loader = DataLoader(
                ds,
                batch_size=batch_size,
                shuffle=False,
                collate_fn=collator,
                num_workers=NUM_WORKERS,
                pin_memory=torch.cuda.is_available(),
            )

            variant_logits = []
            for batch in tqdm(loader, desc=f"{desc} | prompt={prompt_variant}", unit="batch"):
                batch = move_batch_to_device(batch, target_device)
                with torch.no_grad():
                    with torch.autocast(
                        device_type="cuda",
                        dtype=TORCH_DTYPE,
                        enabled=torch.cuda.is_available(),
                    ):
                        logits = compute_choice_logits(model, batch, choice_token_ids)
                variant_logits.append(logits.float().cpu())

            variant_logits = torch.cat(variant_logits, dim=0)
            logits_sum = variant_logits if logits_sum is None else (logits_sum + variant_logits)

    return logits_sum / max(len(prompt_variant_ids), 1)

def attach_teacher_scores(df: pd.DataFrame, choice_logits: torch.Tensor) -> pd.DataFrame:
    probs = probs_from_logits(choice_logits).numpy()
    out = df.reset_index(drop=True).copy()

    for i, c in enumerate(LETTERS):
        out[f"teacher_p_{c}"] = probs[:, i]

    pred_idx = probs.argmax(axis=1)
    sorted_probs = np.sort(probs, axis=1)
    conf = sorted_probs[:, -1]
    second = sorted_probs[:, -2]
    margin = conf - second
    entropy = -(probs * np.log(probs + 1e-12)).sum(axis=1)

    out["teacher_pred"] = [IDX_TO_LETTER[int(i)] for i in pred_idx]
    out["teacher_conf"] = conf
    out["teacher_margin"] = margin
    out["teacher_entropy"] = entropy
    out["teacher_agree"] = (out["teacher_pred"] == out["answer"].astype(str).str.lower()).astype(int)

    return out

def build_noise_robust_targets(df: pd.DataFrame) -> pd.DataFrame:
    out = df.reset_index(drop=True).copy()
    probs = out[[f"teacher_p_{c}" for c in LETTERS]].to_numpy(dtype=np.float32)
    gold_idx = out["answer"].astype(str).str.lower().map(LETTER_TO_IDX).to_numpy()

    gold_prob = probs[np.arange(len(out)), gold_idx]
    pred_idx = probs.argmax(axis=1)
    pred_conf = probs.max(axis=1)
    second = np.sort(probs, axis=1)[:, -2]
    margin = pred_conf - second
    agree = pred_idx == gold_idx

    # gold와 teacher를 부드럽게 섞되,
    # high-confidence disagreement는 강하게 down-weight
    gold_mix = np.clip(0.15 + 0.75 * gold_prob + 0.15 * agree.astype(np.float32), 0.15, 0.95)
    sample_weight = np.clip(0.10 + 0.90 * gold_prob + 0.20 * agree.astype(np.float32), 0.10, 1.00)

    hard_noisy = (~agree) & (pred_conf >= 0.85) & (margin >= 0.20)
    sample_weight[hard_noisy] = np.minimum(sample_weight[hard_noisy], 0.20)
    gold_mix[hard_noisy] = np.minimum(gold_mix[hard_noisy], 0.25)

    out["gold_mix"] = gold_mix.astype(np.float32)
    out["sample_weight"] = sample_weight.astype(np.float32)
    out["phase1_keep"] = (sample_weight >= 0.65).astype(int)
    out["eval_keep"] = (sample_weight >= 0.30).astype(int)

    target_mat = []
    for i in range(len(out)):
        onehot = np.eye(4, dtype=np.float32)[gold_idx[i]]
        target = gold_mix[i] * onehot + (1.0 - gold_mix[i]) * probs[i]
        target = label_smooth_probs(target.astype(np.float32), LABEL_SMOOTHING)
        target_mat.append(target)

    target_mat = np.stack(target_mat)
    for i, c in enumerate(LETTERS):
        out[f"target_p_{c}"] = target_mat[:, i]

    return out

# py3.10 이하 호환용
try:
    from contextlib import nullcontext
except Exception:
    class nullcontext:
        def __enter__(self): return self
        def __exit__(self, *excinfo): return False

def save_text(path: str, text: str):
    with open(path, "w", encoding="utf-8") as f:
        f.write(str(text))

def read_text(path: str, default: str = "") -> str:
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()


PROMPT_TEMPLATES[2] = lambda q, a, b, c, d: (
    "사진과 질문을 함께 보고 4개 보기 중 하나를 고르세요.\n"
    "설명 없이 a, b, c, d 중 하나만 출력하세요.\n\n"
    f"질문: {q}\n"
    f"a. {a}\n"
    f"b. {b}\n"
    f"c. {c}\n"
    f"d. {d}\n\n"
    "선택:"
)

def sanitize_name(x: str) -> str:
    return re.sub(r"[^a-zA-Z0-9]+", "_", str(x)).strip("_")

def unique_keep_order(seq: List[Any]) -> List[Any]:
    out = []
    seen = set()
    for x in seq:
        if x in seen:
            continue
        seen.add(x)
        out.append(x)
    return out

def list_checkpoint_dirs(run_dir: str, pattern: str = LEGACY_CKPT_PATTERN) -> List[str]:
    import glob
    dirs = sorted([p for p in glob.glob(os.path.join(run_dir, pattern)) if os.path.isdir(p)])
    return dirs

def resolve_resume_checkpoint(run_dir: str, fallback: Optional[str] = None) -> Optional[str]:
    latest_txt = os.path.join(run_dir, "latest_checkpoint.txt")
    latest_ckpt = read_text(latest_txt, default="")
    if latest_ckpt and os.path.exists(latest_ckpt):
        return latest_ckpt
    if fallback and os.path.exists(fallback):
        return fallback
    return None

def save_json(path: str, obj: Dict[str, Any]):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def maybe_patch_gemma4_clippable_linear():
    """
    Gemma 4 vision tower의 Gemma4ClippableLinear가 일부 PEFT 조합에서
    LoRA 삽입 시 문제가 날 수 있어 필요 시 monkey patch.
    """
    try:
        import torch.nn as nn
        from transformers.models.gemma4 import modeling_gemma4
    except Exception as e:
        print("Patch skipped:", repr(e))
        return

    class PatchedClippableLinear(nn.Linear):
        def __init__(self, config, in_features, out_features):
            nn.Linear.__init__(self, in_features, out_features, bias=False)
            self.use_clipped_linears = getattr(config, "use_clipped_linears", False)
            if self.use_clipped_linears:
                self.register_buffer("input_min", torch.tensor(-float("inf")))
                self.register_buffer("input_max", torch.tensor(float("inf")))
                self.register_buffer("output_min", torch.tensor(-float("inf")))
                self.register_buffer("output_max", torch.tensor(float("inf")))

        def forward(self, x):
            if self.use_clipped_linears:
                x = torch.clamp(x, self.input_min, self.input_max)
            out = nn.Linear.forward(self, x)
            if self.use_clipped_linears:
                out = torch.clamp(out, self.output_min, self.output_max)
            return out

    modeling_gemma4.Gemma4ClippableLinear = PatchedClippableLinear
    print("Gemma4ClippableLinear patch applied.")

def make_4bit_bnb_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=TORCH_DTYPE,
        bnb_4bit_quant_storage=TORCH_DTYPE,
    )

def load_quantized_base_model(model_id: str = MODEL_ID):
    if ENABLE_VISION_TOWER_LORA:
        maybe_patch_gemma4_clippable_linear()
    model = AutoVisionTextModel.from_pretrained(
        model_id,
        quantization_config=make_4bit_bnb_config(),
        torch_dtype=TORCH_DTYPE,
        device_map="auto",
        attn_implementation=ATTN_IMPLEMENTATION,
        low_cpu_mem_usage=True,
    )
    return model

def build_choice_token_ids(processor) -> torch.Tensor:
    ids_out = []
    for c in LETTERS:
        ids = processor.tokenizer(c, add_special_tokens=False)["input_ids"]
        print(f"choice token {c}: {ids}")
        if len(ids) != 1:
            raise ValueError(
                f"Choice letter '{c}' is not a single token: {ids}. "
                "이 노트북은 a/b/c/d 한 글자 분류를 전제로 합니다."
            )
        ids_out.append(ids[0])
    return torch.tensor(ids_out, dtype=torch.long)

def ensure_base_teacher_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.reset_index(drop=True).copy()
    for c in LETTERS:
        teacher_col = f"teacher_p_{c}"
        base_col = f"base_p_{c}"
        if teacher_col in out.columns and base_col not in out.columns:
            out[base_col] = out[teacher_col].astype(np.float32)
    copy_pairs = [
        ("teacher_pred", "base_pred"),
        ("teacher_conf", "base_conf"),
        ("teacher_margin", "base_margin"),
        ("teacher_entropy", "base_entropy"),
        ("teacher_agree", "base_agree"),
    ]
    for src, dst in copy_pairs:
        if src in out.columns and dst not in out.columns:
            out[dst] = out[src]
    return out

def evaluate_logits_bundle(choice_logits: torch.Tensor, df: pd.DataFrame, clean_mask: Optional[np.ndarray] = None) -> Dict[str, float]:
    metrics_all = summarize_logits_against_df(choice_logits, df)
    out = {
        "all_valid_acc": float(metrics_all["raw_acc"]),
        "all_valid_weighted_acc": float(metrics_all["weighted_acc"]),
        "all_valid_ce": float(metrics_all["weighted_ce"]),
    }

    if clean_mask is not None and clean_mask.sum() > 0:
        clean_df = df.iloc[np.where(clean_mask)[0]].reset_index(drop=True)
        clean_logits = choice_logits[torch.from_numpy(clean_mask.astype(bool))]
        metrics_clean = summarize_logits_against_df(clean_logits, clean_df)
        out.update({
            "clean_valid_acc": float(metrics_clean["raw_acc"]),
            "clean_valid_weighted_acc": float(metrics_clean["weighted_acc"]),
            "clean_valid_ce": float(metrics_clean["weighted_ce"]),
        })
    else:
        out.update({
            "clean_valid_acc": float("nan"),
            "clean_valid_weighted_acc": float("nan"),
            "clean_valid_ce": float("nan"),
        })
    return out

def metric_rank_tuple(metrics: Dict[str, float]) -> Tuple[float, float, float, float]:
    return (
        float(metrics.get("all_valid_acc", -1.0)),
        float(metrics.get("clean_valid_acc", -1.0)),
        -float(metrics.get("all_valid_ce", 1e9)),
        -float(metrics.get("clean_valid_ce", 1e9)),
    )

def better_metrics(a: Dict[str, float], b: Optional[Dict[str, float]], eps: float = 1e-8) -> bool:
    if b is None:
        return True
    ta = metric_rank_tuple(a)
    tb = metric_rank_tuple(b)
    for xa, xb in zip(ta, tb):
        if xa > xb + eps:
            return True
        if xa < xb - eps:
            return False
    return False

def greedy_search_logits_map(
    logits_map: Dict[str, torch.Tensor],
    df: pd.DataFrame,
    clean_mask: Optional[np.ndarray] = None,
    max_models: int = 4,
    alpha_grid: Optional[List[float]] = None,
):
    if not logits_map:
        raise ValueError("logits_map is empty")

    alpha_grid = alpha_grid or GREEDY_ALPHA_GRID
    singles = {}
    for name, logits in logits_map.items():
        singles[name] = evaluate_logits_bundle(logits, df, clean_mask=clean_mask)

    best_single_name = None
    best_single_metrics = None
    for name, metrics in singles.items():
        if better_metrics(metrics, best_single_metrics):
            best_single_name = name
            best_single_metrics = metrics

    selected = [best_single_name]
    weights = {best_single_name: 1.0}
    ensemble_logits = logits_map[best_single_name].clone()
    trace_rows = [{"round": 0, "selected": best_single_name, "alpha": 1.0, **best_single_metrics}]

    for round_idx in range(1, max_models):
        round_best = None
        round_best_logits = None
        round_best_weights = None
        for name, candidate_logits in logits_map.items():
            if name in selected:
                continue
            for alpha in alpha_grid:
                mixed_logits = (1.0 - alpha) * ensemble_logits + alpha * candidate_logits
                mixed_metrics = evaluate_logits_bundle(mixed_logits, df, clean_mask=clean_mask)
                candidate_info = {
                    "name": name,
                    "alpha": float(alpha),
                    "metrics": mixed_metrics,
                }
                if better_metrics(candidate_info["metrics"], round_best["metrics"] if round_best is not None else None):
                    round_best = candidate_info
                    round_best_logits = mixed_logits
                    new_weights = {k: v * (1.0 - alpha) for k, v in weights.items()}
                    new_weights[name] = new_weights.get(name, 0.0) + float(alpha)
                    round_best_weights = new_weights

        if round_best is None:
            break

        current_metrics = evaluate_logits_bundle(ensemble_logits, df, clean_mask=clean_mask)
        if not better_metrics(round_best["metrics"], current_metrics):
            break

        selected.append(round_best["name"])
        weights = round_best_weights
        ensemble_logits = round_best_logits
        trace_rows.append({
            "round": round_idx,
            "selected": round_best["name"],
            "alpha": round_best["alpha"],
            **round_best["metrics"],
        })

    weight_sum = sum(weights.values())
    if weight_sum > 0:
        weights = {k: float(v / weight_sum) for k, v in weights.items()}
    return selected, weights, ensemble_logits, pd.DataFrame(trace_rows)

def attach_human_teacher_probs(df: pd.DataFrame, probs: np.ndarray, prefix: str = "human") -> pd.DataFrame:
    out = df.reset_index(drop=True).copy()
    for i, c in enumerate(LETTERS):
        out[f"{prefix}_p_{c}"] = probs[:, i].astype(np.float32)
    pred_idx = probs.argmax(axis=1)
    sorted_probs = np.sort(probs, axis=1)
    out[f"{prefix}_pred"] = [IDX_TO_LETTER[int(i)] for i in pred_idx]
    out[f"{prefix}_conf"] = sorted_probs[:, -1].astype(np.float32)
    out[f"{prefix}_margin"] = (sorted_probs[:, -1] - sorted_probs[:, -2]).astype(np.float32)
    out[f"{prefix}_entropy"] = (-(probs * np.log(probs + 1e-12)).sum(axis=1)).astype(np.float32)
    return out

def weighted_average_probs_from_logits_map(logits_map: Dict[str, torch.Tensor], weights: Dict[str, float]) -> np.ndarray:
    if not weights:
        raise ValueError("weights is empty")
    probs_accum = None
    for name, w in weights.items():
        probs = probs_from_logits(logits_map[name]).cpu().numpy()
        if probs_accum is None:
            probs_accum = float(w) * probs
        else:
            probs_accum += float(w) * probs
    probs_accum = probs_accum / np.clip(probs_accum.sum(axis=1, keepdims=True), 1e-8, None)
    return probs_accum.astype(np.float32)

def build_human_bias_targets(df: pd.DataFrame, human_teacher_weight: float = HUMAN_TEACHER_WEIGHT) -> pd.DataFrame:
    out = ensure_base_teacher_columns(df)
    base_probs = out[[f"base_p_{c}" for c in LETTERS]].to_numpy(dtype=np.float32)
    human_probs = out[[f"human_p_{c}" for c in LETTERS]].to_numpy(dtype=np.float32)

    gold_idx = out["answer"].astype(str).str.lower().map(LETTER_TO_IDX).to_numpy()
    row_idx = np.arange(len(out))

    base_pred = base_probs.argmax(axis=1)
    human_pred = human_probs.argmax(axis=1)
    base_conf = base_probs.max(axis=1)
    human_conf = human_probs.max(axis=1)
    base_gold = base_probs[row_idx, gold_idx]
    human_gold = human_probs[row_idx, gold_idx]

    base_agree = base_pred == gold_idx
    human_agree = human_pred == gold_idx
    both_agree = base_agree & human_agree
    both_disagree = (~base_agree) & (~human_agree)
    human_only_agree = human_agree & (~base_agree)
    base_only_agree = base_agree & (~human_agree)

    gold_mix = 0.25 + 0.40 * human_gold + 0.15 * base_gold + 0.10 * human_agree.astype(np.float32) + 0.05 * base_agree.astype(np.float32)
    sample_weight = 0.20 + 0.45 * human_gold + 0.15 * base_gold + 0.15 * human_agree.astype(np.float32) + 0.05 * base_agree.astype(np.float32)

    strong_both_disagree = both_disagree & (human_conf >= 0.75) & (base_conf >= 0.80)
    gold_mix[strong_both_disagree] = np.minimum(gold_mix[strong_both_disagree], 0.20)
    sample_weight[strong_both_disagree] = np.minimum(sample_weight[strong_both_disagree], 0.25)

    gold_mix[human_only_agree] = np.maximum(gold_mix[human_only_agree], 0.55)
    sample_weight[human_only_agree] = np.maximum(sample_weight[human_only_agree], 0.55)

    gold_mix[base_only_agree] = np.clip(gold_mix[base_only_agree], 0.30, 0.75)
    sample_weight[base_only_agree] = np.clip(sample_weight[base_only_agree], 0.30, 0.75)

    gold_mix[both_agree] = np.maximum(gold_mix[both_agree], 0.70)
    sample_weight[both_agree] = np.maximum(sample_weight[both_agree], 0.80)

    gold_mix = np.clip(gold_mix, 0.15, 0.92).astype(np.float32)
    sample_weight = np.clip(sample_weight, 0.10, 1.00).astype(np.float32)

    fused_teacher = human_teacher_weight * human_probs + (1.0 - human_teacher_weight) * base_probs
    fused_teacher = fused_teacher / np.clip(fused_teacher.sum(axis=1, keepdims=True), 1e-8, None)

    target_mat = []
    for i in range(len(out)):
        onehot = np.eye(4, dtype=np.float32)[gold_idx[i]]
        target = gold_mix[i] * onehot + (1.0 - gold_mix[i]) * fused_teacher[i]
        target = label_smooth_probs(target.astype(np.float32), LABEL_SMOOTHING)
        target = target / np.clip(target.sum(), 1e-8, None)
        target_mat.append(target.astype(np.float32))
    target_mat = np.stack(target_mat)

    for i, c in enumerate(LETTERS):
        out[f"target_p_{c}"] = target_mat[:, i].astype(np.float32)

    out["gold_mix"] = gold_mix
    out["sample_weight"] = sample_weight
    out["phase1_keep"] = (sample_weight >= 0.60).astype(int)
    out["eval_keep"] = (sample_weight >= 0.25).astype(int)
    out["human_gold_prob"] = human_gold.astype(np.float32)
    out["base_gold_prob"] = base_gold.astype(np.float32)
    out["human_agree"] = human_agree.astype(int)
    out["base_agree"] = base_agree.astype(int)
    out["both_disagree"] = both_disagree.astype(int)
    return out

def make_adapter_key(adapter_dir: str, prefix: str = "adapter") -> str:
    path = Path(adapter_dir)
    return sanitize_name(f"{prefix}__{path.parent.name}__{path.name}")

def build_multi_adapter_model(base_model, adapter_dirs: List[str], prefix: str = "adapter"):
    adapter_dirs = [p for p in adapter_dirs if p and os.path.isdir(p)]
    adapter_dirs = unique_keep_order(adapter_dirs)
    if not adapter_dirs:
        return None, {}

    model = None
    registry = {}
    for adapter_dir in adapter_dirs:
        key = make_adapter_key(adapter_dir, prefix=prefix)
        adapter_name = key
        if model is None:
            try:
                model = PeftModel.from_pretrained(base_model, adapter_dir, adapter_name=adapter_name, is_trainable=False)
            except TypeError:
                model = PeftModel.from_pretrained(base_model, adapter_dir, is_trainable=False)
        else:
            try:
                model.load_adapter(adapter_dir, adapter_name=adapter_name, is_trainable=False)
            except TypeError:
                model.load_adapter(adapter_dir, adapter_name=adapter_name)

        registry[key] = {
            "path": adapter_dir,
            "adapter_name": adapter_name,
            "kind": "adapter",
        }

    model.eval()
    return model, registry

def score_registry_entry_logits(
    model,
    processor,
    df: pd.DataFrame,
    registry_entry: Dict[str, Any],
    choice_token_ids: torch.Tensor,
    prompt_variant_ids: List[int],
    desc: str,
):
    if registry_entry["kind"] == "base":
        return score_df_choice_logits(
            model,
            processor,
            df,
            prompt_variant_ids=prompt_variant_ids,
            choice_token_ids=choice_token_ids,
            batch_size=1,
            desc=desc,
            disable_adapter=True,
        )

    model.set_adapter(registry_entry["adapter_name"])
    return score_df_choice_logits(
        model,
        processor,
        df,
        prompt_variant_ids=prompt_variant_ids,
        choice_token_ids=choice_token_ids,
        batch_size=1,
        desc=desc,
        disable_adapter=False,
    )

def predict_row_with_registry_ensemble(
    model,
    processor,
    registry: Dict[str, Dict[str, Any]],
    model_weights: Dict[str, float],
    row_like: Any,
    choice_token_ids: torch.Tensor,
    prompt_variant_ids: List[int],
    num_option_permutations: int,
):
    if not model_weights:
        raise ValueError("model_weights is empty")

    samples = build_inference_samples_for_row(
        row_like,
        prompt_variant_ids=prompt_variant_ids,
        num_option_permutations=num_option_permutations,
    )

    final_logits = None
    for key, weight in model_weights.items():
        entry = registry[key]
        if entry["kind"] == "base":
            ctx = model.disable_adapter() if hasattr(model, "disable_adapter") else nullcontext()
            with ctx:
                logits_list = score_inference_samples(model, processor, samples, choice_token_ids)
        else:
            model.set_adapter(entry["adapter_name"])
            logits_list = score_inference_samples(model, processor, samples, choice_token_ids)

        remapped_logits = []
        for i, sample in enumerate(samples):
            cur_logits = remap_logits_current_to_original(logits_list[i], sample["new_to_orig"])
            remapped_logits.append(cur_logits)
        model_logits = torch.stack(remapped_logits, dim=0).mean(dim=0)

        if final_logits is None:
            final_logits = float(weight) * model_logits
        else:
            final_logits = final_logits + float(weight) * model_logits

    pred_idx = int(final_logits.argmax().item())
    pred_letter = IDX_TO_LETTER[pred_idx]
    pred_probs = F.softmax(final_logits, dim=-1).numpy()
    return {
        "pred_letter": pred_letter,
        "logits": final_logits,
        "probs": pred_probs,
    }


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
고정 split 불러옴: /content/drive/MyDrive/gemma4_31b_it_mcqa_lora_v5
MODEL_ID: google/gemma-4-31B-it
USE_BF16: True
train_pool size: 4668
valid size: 405
test size: 5074


# Legacy checkpoint 재평가 + human teacher target 생성

여기서는 다음을 수행합니다.

1. v5에서 저장된 **epoch 1~5 checkpoint**를 모두 찾음
2. **full valid raw accuracy** 기준으로 어떤 checkpoint 조합이 사람식 라벨을 가장 잘 따라가는지 greedy search
3. 선택된 checkpoint ensemble을 **human teacher**로 사용
4. base zero-shot teacher와 human teacher를 섞어 **새 train/valid soft target** 생성

중요한 점:
- v5는 clean-valid 중심으로 best checkpoint를 골랐지만,
  지금 목표는 **테스트의 인간식 라벨 분포**이므로 **all-valid raw accuracy**를 우선합니다.


In [6]:

import glob

legacy_history_path = os.path.join(PREV_RUN_DIR, "train_history.csv")
if os.path.exists(legacy_history_path):
    legacy_history_df = pd.read_csv(legacy_history_path)
    print("[legacy train history]")
    display(legacy_history_df)
else:
    legacy_history_df = None
    print("legacy train history not found:", legacy_history_path)

scoring_processor = AutoProcessor.from_pretrained(MODEL_ID)
choice_token_ids = build_choice_token_ids(scoring_processor)

scoring_base_model = load_quantized_base_model(MODEL_ID)
scoring_base_model.eval()
target_device = next(scoring_base_model.parameters()).device
print("Scoring device:", target_device)

base_teacher_train_cache = None
base_teacher_valid_cache = None
for cache_dir in [PREV_RUN_DIR, RUN_DIR]:
    cand_train = os.path.join(cache_dir, "teacher_scored_train_pool.csv")
    cand_valid = os.path.join(cache_dir, "teacher_scored_valid.csv")
    if os.path.exists(cand_train) and os.path.exists(cand_valid):
        base_teacher_train_cache = cand_train
        base_teacher_valid_cache = cand_valid
        break

if base_teacher_train_cache and base_teacher_valid_cache:
    train_base_df = ensure_base_teacher_columns(pd.read_csv(base_teacher_train_cache))
    valid_base_df = ensure_base_teacher_columns(pd.read_csv(base_teacher_valid_cache))
    print("base teacher cache 불러옴:", os.path.dirname(base_teacher_train_cache))
else:
    print("base teacher cache 생성 시작...")
    train_teacher_logits = score_df_choice_logits(
        scoring_base_model,
        scoring_processor,
        train_pool_df,
        prompt_variant_ids=PROMPT_VARIANT_IDS_EVAL,
        choice_token_ids=choice_token_ids,
        batch_size=TEACHER_BATCH_SIZE,
        desc="base train teacher",
        disable_adapter=False,
    )
    valid_teacher_logits = score_df_choice_logits(
        scoring_base_model,
        scoring_processor,
        valid_df,
        prompt_variant_ids=PROMPT_VARIANT_IDS_EVAL,
        choice_token_ids=choice_token_ids,
        batch_size=TEACHER_BATCH_SIZE,
        desc="base valid teacher",
        disable_adapter=False,
    )
    train_base_df = ensure_base_teacher_columns(build_noise_robust_targets(attach_teacher_scores(train_pool_df, train_teacher_logits)))
    valid_base_df = ensure_base_teacher_columns(build_noise_robust_targets(attach_teacher_scores(valid_df, valid_teacher_logits)))
    base_teacher_train_cache = os.path.join(RUN_DIR, "teacher_scored_train_pool.csv")
    base_teacher_valid_cache = os.path.join(RUN_DIR, "teacher_scored_valid.csv")
    train_base_df.to_csv(base_teacher_train_cache, index=False)
    valid_base_df.to_csv(base_teacher_valid_cache, index=False)
    print("base teacher cache 저장:", RUN_DIR)

legacy_ckpt_dirs = list_checkpoint_dirs(PREV_RUN_DIR, LEGACY_CKPT_PATTERN)
print("legacy checkpoints found:", len(legacy_ckpt_dirs))
for ckpt_dir in legacy_ckpt_dirs:
    print(" -", ckpt_dir)
if not legacy_ckpt_dirs:
    raise ValueError(f"No legacy checkpoints found under {PREV_RUN_DIR}")

legacy_model, legacy_registry = build_multi_adapter_model(scoring_base_model, legacy_ckpt_dirs, prefix="legacy")
target_device = next(legacy_model.parameters()).device
print("Legacy multi-adapter model ready on:", target_device)

legacy_clean_mask = valid_base_df["eval_keep"].to_numpy().astype(bool) if "eval_keep" in valid_base_df.columns else None
legacy_valid_logits_map = {}
legacy_rows = []

for key, entry in legacy_registry.items():
    cache_path = os.path.join(RUN_DIR, f"legacy_valid_logits__{key}.pt")
    if os.path.exists(cache_path):
        logits = torch.load(cache_path, map_location="cpu")
        print("valid logits cache hit:", key)
    else:
        logits = score_registry_entry_logits(
            legacy_model,
            scoring_processor,
            valid_base_df,
            entry,
            choice_token_ids=choice_token_ids,
            prompt_variant_ids=PROMPT_VARIANT_IDS_EVAL,
            desc=f"legacy valid | {key}",
        )
        torch.save(logits.float().cpu(), cache_path)
        print("valid logits cache saved:", cache_path)

    logits = logits.float().cpu()
    legacy_valid_logits_map[key] = logits
    metrics = evaluate_logits_bundle(logits, valid_base_df, clean_mask=legacy_clean_mask)
    legacy_rows.append({
        "key": key,
        "path": entry["path"],
        **metrics,
    })

legacy_valid_table = pd.DataFrame(legacy_rows).sort_values(
    ["all_valid_acc", "clean_valid_acc", "all_valid_ce"],
    ascending=[False, False, True],
).reset_index(drop=True)
print("\n[legacy checkpoint valid ranking]")
display(legacy_valid_table)

human_selected_keys, human_weights, human_valid_ensemble_logits, human_search_trace = greedy_search_logits_map(
    legacy_valid_logits_map,
    valid_base_df,
    clean_mask=legacy_clean_mask,
    max_models=MAX_HUMAN_TEACHER_MODELS,
    alpha_grid=GREEDY_ALPHA_GRID,
)
print("\n[human teacher ensemble selected]")
print("selected keys:", human_selected_keys)
print("weights:", human_weights)
display(human_search_trace)

human_teacher_selection = [
    {
        "key": key,
        "path": legacy_registry[key]["path"],
        "weight": float(human_weights[key]),
    }
    for key in human_selected_keys
]
save_json(os.path.join(RUN_DIR, "human_teacher_selection.json"), {
    "selected": human_teacher_selection,
    "weights": {k: float(v) for k, v in human_weights.items()},
})

selected_train_logits_map = {}
selected_valid_logits_map = {}
for key in human_selected_keys:
    entry = legacy_registry[key]
    for split_name, df_cur in [("train", train_base_df), ("valid", valid_base_df)]:
        cache_path = os.path.join(RUN_DIR, f"legacy_{split_name}_logits__{key}.pt")
        if os.path.exists(cache_path):
            logits = torch.load(cache_path, map_location="cpu").float()
            print(f"{split_name} logits cache hit:", key)
        else:
            logits = score_registry_entry_logits(
                legacy_model,
                scoring_processor,
                df_cur,
                entry,
                choice_token_ids=choice_token_ids,
                prompt_variant_ids=PROMPT_VARIANT_IDS_EVAL,
                desc=f"legacy {split_name} | {key}",
            ).float().cpu()
            torch.save(logits, cache_path)
            print(f"{split_name} logits cache saved:", cache_path)

        if split_name == "train":
            selected_train_logits_map[key] = logits
        else:
            selected_valid_logits_map[key] = logits

human_train_probs = weighted_average_probs_from_logits_map(selected_train_logits_map, human_weights)
human_valid_probs = weighted_average_probs_from_logits_map(selected_valid_logits_map, human_weights)

train_pool_scored_df = build_human_bias_targets(
    attach_human_teacher_probs(train_base_df, human_train_probs),
    human_teacher_weight=HUMAN_TEACHER_WEIGHT,
)
valid_scored_df = build_human_bias_targets(
    attach_human_teacher_probs(valid_base_df, human_valid_probs),
    human_teacher_weight=HUMAN_TEACHER_WEIGHT,
)

train_human_cache = os.path.join(RUN_DIR, "human_bias_train_pool.csv")
valid_human_cache = os.path.join(RUN_DIR, "human_bias_valid.csv")
train_pool_scored_df.to_csv(train_human_cache, index=False)
valid_scored_df.to_csv(valid_human_cache, index=False)
print("human-bias target cache saved:")
print(" -", train_human_cache)
print(" -", valid_human_cache)

stage1_train_df = train_pool_scored_df[train_pool_scored_df["phase1_keep"] == 1].reset_index(drop=True)
clean_valid_df = valid_scored_df[valid_scored_df["eval_keep"] == 1].reset_index(drop=True)

print("\n[target stats]")
print("stage1_train size:", len(stage1_train_df))
print("full_train size:", len(train_pool_scored_df))
print("clean_valid size:", len(clean_valid_df))
print("valid size:", len(valid_scored_df))
print("mean sample_weight (train):", float(train_pool_scored_df["sample_weight"].mean()))
print("mean gold_mix (train):", float(train_pool_scored_df["gold_mix"].mean()))
print("human agree ratio (train):", float(train_pool_scored_df["human_agree"].mean()))
print("base agree ratio (train):", float(train_pool_scored_df["base_agree"].mean()))

display(
    train_pool_scored_df[
        [
            "id", "answer", "human_pred", "base_pred", "human_conf", "base_conf",
            "human_gold_prob", "base_gold_prob", "gold_mix", "sample_weight", "both_disagree",
        ]
    ].head(12)
)

del legacy_model
del scoring_base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


[legacy train history]


,epoch,phase_name,train_loss,train_weighted_acc,clean_valid_acc,clean_valid_weighted_ce,all_valid_acc,all_valid_weighted_ce,best_clean_valid_acc,best_all_valid_acc,best_clean_valid_ce,patience_counter,train_size
0,1,stage1_clean_warmup,0.563763,0.931819,0.959877,0.544096,0.933333,0.639302,0.959877,0.933333,0.544096,0,2749
1,2,stage2_full_weighted,0.924576,0.945980,0.962963,0.644670,0.943210,0.712638,0.962963,0.943210,0.644670,0,4668
2,3,stage2_full_weighted,0.893189,0.961757,0.935185,0.611408,0.916049,0.681712,0.962963,0.943210,0.644670,1,4668
3,4,stage2_full_weighted,0.869234,0.972657,0.966049,0.572940,0.948148,0.640791,0.966049,0.948148,0.572940,0,4668
4,5,stage2_full_weighted,0.851889,0.977405,0.966049,0.568735,0.945679,0.636854,0.966049,0.948148,0.572940,1,4668
5,6,stage2_full_weighted,0.863970,0.972964,0.969136,0.581414,0.943210,0.644239,0.969136,0.943210,0.581414,0,4668


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

,key,path,all_valid_acc,all_valid_weighted_acc,all_valid_ce,clean_valid_acc,clean_valid_weighted_acc,clean_valid_ce
0,legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_e...,/content/drive/MyDrive/gemma4_31b_it_mcqa_lora...,0.948148,0.964902,0.640100,0.966049,0.970205,0.572213
1,legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_e...,/content/drive/MyDrive/gemma4_31b_it_mcqa_lora...,0.945679,0.964005,0.643832,0.969136,0.970756,0.580851
2,legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_e...,/content/drive/MyDrive/gemma4_31b_it_mcqa_lora...,0.945679,0.964004,0.636006,0.966049,0.970270,0.567764
3,legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_e...,/content/drive/MyDrive/gemma4_31b_it_mcqa_lora...,0.943210,0.956305,0.712370,0.962963,0.961998,0.644420
4,legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_e...,/content/drive/MyDrive/gemma4_31b_it_mcqa_lora...,0.938272,0.964826,0.638472,0.966049,0.973337,0.543650
5,legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_e...,/content/drive/MyDrive/gemma4_31b_it_mcqa_lora...,0.918519,0.940453,0.681074,0.938272,0.945802,0.610783



[human teacher ensemble selected]
selected keys: ['legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_epoch_04', 'legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_epoch_02', 'legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_epoch_05']
weights: {'legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_epoch_04': 0.45499999999999996, 'legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_epoch_02': 0.195, 'legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_epoch_05': 0.35}


,round,selected,alpha,all_valid_acc,all_valid_weighted_acc,all_valid_ce,clean_valid_acc,clean_valid_weighted_acc,clean_valid_ce
0,0,legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_e...,1.00,0.948148,0.964902,0.640100,0.966049,0.970205,0.572213
1,1,legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_e...,0.30,0.958025,0.968964,0.654264,0.972222,0.973199,0.586315
2,2,legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_e...,0.35,0.958025,0.973769,0.645750,0.978395,0.979676,0.577691


train logits cache hit: legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_epoch_04
valid logits cache hit: legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_epoch_04
train logits cache hit: legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_epoch_02
valid logits cache hit: legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_epoch_02
train logits cache hit: legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_epoch_05
valid logits cache hit: legacy_gemma4_31b_it_mcqa_lora_v5_checkpoint_epoch_05
human-bias target cache saved:
 - /content/drive/MyDrive/gemma4_31b_it_mcqa_lora_v6_human_bias/human_bias_train_pool.csv
 - /content/drive/MyDrive/gemma4_31b_it_mcqa_lora_v6_human_bias/human_bias_valid.csv

[target stats]
stage1_train size: 4191
full_train size: 4668
clean_valid size: 405
valid size: 405
mean sample_weight (train): 0.7677995562553406
mean gold_mix (train): 0.7331530451774597
human agree ratio (train): 0.9400171379605827
base agree ratio (train): 0.6107540702656384


,id,answer,human_pred,base_pred,human_conf,base_conf,human_gold_prob,base_gold_prob,gold_mix,sample_weight,both_disagree
0,train_3868.jpg,d,d,d,0.833580,0.795014,0.833580,0.795014,0.852684,0.894363,0
1,train_0747.jpg,a,a,a,0.768870,0.809981,0.768870,0.809981,0.829045,0.867489,0
2,train_1262.jpg,d,d,d,0.760294,0.935094,0.760294,0.935094,0.844382,0.882396,0
3,train_3586.jpg,d,d,a,0.728664,0.798048,0.728664,0.083459,0.653985,0.690418,0
4,train_2599.jpg,a,a,a,0.774204,0.691687,0.774204,0.691687,0.813435,0.852145,0
5,train_3209.jpg,b,b,a,0.728141,0.781581,0.728141,0.118695,0.659061,0.695468,0
6,train_4995.jpg,c,c,c,0.797570,0.717883,0.797570,0.717883,0.826710,0.866589,0
7,train_0110.jpg,a,a,a,0.787336,0.752232,0.787336,0.752232,0.827769,0.867136,0
8,train_2989.jpg,a,a,d,0.260262,0.544420,0.260262,0.206639,0.550000,0.550000,0
9,train_4029.jpg,a,a,a,0.664739,0.751683,0.664739,0.751683,0.778648,0.811885,0


# Trainable model 초기화

- v6를 이미 일부 돌린 상태면 `RUN_DIR/latest_checkpoint.txt`에서 resume
- 그렇지 않으면 **v5 epoch5 checkpoint**를 초기값으로 사용
- optimizer / scheduler는 **v6 설정으로 새로 시작**


In [7]:

# ---------------------------------
# fresh base model -> kbit training 준비
# ---------------------------------
base_model = load_quantized_base_model(MODEL_ID)
base_model.config.use_cache = False
if hasattr(base_model.config, "text_config"):
    base_model.config.text_config.use_cache = False

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# ---------------------------------
# target modules
# ---------------------------------
if ENABLE_VISION_TOWER_LORA:
    lora_target_modules = "all-linear"
else:
    lora_target_modules = (
        r".*(language_model\.layers\.\d+\.(self_attn\.(q_proj|k_proj|v_proj|o_proj)|"
        r"mlp\.(gate_proj|up_proj|down_proj))|embed_vision\.embedding_projection)$"
    )

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=lora_target_modules,
    task_type="CAUSAL_LM",
    use_rslora=USE_RSLORA,
    ensure_weight_tying=True,
)

resume_ckpt_current_run = resolve_resume_checkpoint(RUN_DIR)
init_ckpt = INIT_FROM_CHECKPOINT if INIT_FROM_CHECKPOINT and os.path.exists(INIT_FROM_CHECKPOINT) else None

start_epoch = 1
best_all_valid_acc = -1.0
best_clean_valid_acc = -1.0
best_all_valid_ce = float("inf")
best_clean_valid_ce = float("inf")
patience_counter = 0
history = []
loaded_from = None
loaded_for_resume = False

if resume_ckpt_current_run is not None:
    loaded_from = resume_ckpt_current_run
    loaded_for_resume = True
    model = PeftModel.from_pretrained(
        base_model,
        resume_ckpt_current_run,
        is_trainable=True,
    )
    processor = AutoProcessor.from_pretrained(resume_ckpt_current_run)

    trainer_state_path = os.path.join(resume_ckpt_current_run, "trainer_state.pt")
    if os.path.exists(trainer_state_path):
        resume_state = torch.load(trainer_state_path, map_location="cpu")
        start_epoch = int(resume_state.get("epoch", 0)) + 1
        best_all_valid_acc = float(resume_state.get("best_all_valid_acc", -1.0))
        best_clean_valid_acc = float(resume_state.get("best_clean_valid_acc", -1.0))
        best_all_valid_ce = float(resume_state.get("best_all_valid_ce", float("inf")))
        best_clean_valid_ce = float(resume_state.get("best_clean_valid_ce", float("inf")))
        patience_counter = int(resume_state.get("patience_counter", 0))
        history = resume_state.get("history", [])
else:
    if init_ckpt is not None:
        loaded_from = init_ckpt
        model = PeftModel.from_pretrained(
            base_model,
            init_ckpt,
            is_trainable=True,
        )
        processor = AutoProcessor.from_pretrained(init_ckpt)
        print("Initialize from legacy checkpoint:", init_ckpt)
    else:
        loaded_from = MODEL_ID
        model = get_peft_model(base_model, lora_config)
        processor = AutoProcessor.from_pretrained(MODEL_ID)
        if USE_LOFTQ_INIT and replace_lora_weights_loftq is not None:
            try:
                replace_lora_weights_loftq(model)
                print("LoftQ init applied.")
            except Exception as e:
                print("LoftQ init skipped:", repr(e))

model.print_trainable_parameters()
choice_token_ids = build_choice_token_ids(processor)
target_device = next((p.device for p in model.parameters() if p.requires_grad), next(model.parameters()).device)
print("Training target device:", target_device)
print("loaded_from:", loaded_from)
print("resume current run:", loaded_for_resume)
print("start_epoch:", start_epoch)


Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1310: UserWarning: You have requested `ensure_weight_tying`, but no tied modules are added in either `modules_to_save` or `target_modules`
  warnings.warn(


trainable params: 122,533,888 || all params: 31,395,620,400 || trainable%: 0.3903
choice token a: [236746]
choice token b: [236763]
choice token c: [236755]
choice token d: [236753]
Training target device: cuda:0
loaded_from: /content/drive/MyDrive/gemma4_31b_it_mcqa_lora_v6_human_bias/checkpoint-epoch-04
resume current run: True
start_epoch: 5


# Phase plan / optimizer

짧고 보수적인 continuation만 수행합니다.

- stage1: human-consistent subset으로 1 epoch
- stage2: full human-bias weighted train pool로 2 epoch
- learning rate는 기존보다 낮게 유지


In [8]:

# ---------------------------------
# phase plan
# ---------------------------------
PHASE_PLAN = (
    [("stage1_human_consistent", stage1_train_df)] * NUM_EPOCHS_STAGE1
    + [("stage2_full_human_mix", train_pool_scored_df)] * NUM_EPOCHS_STAGE2
)
TOTAL_EPOCHS = len(PHASE_PLAN)

def build_train_loader(df: pd.DataFrame, epoch: int):
    ds = NoiseRobustVQADataset(
        df,
        train=True,
        option_shuffle_prob=TRAIN_OPTION_SHUFFLE_PROB,
        prompt_variant_ids=PROMPT_VARIANT_IDS_TRAIN,
    )
    ds.set_epoch(epoch)

    g = torch.Generator()
    g.manual_seed(SEED + epoch)

    loader = DataLoader(
        ds,
        batch_size=PER_DEVICE_BATCH_SIZE,
        shuffle=True,
        generator=g,
        collate_fn=ChoiceLogitCollator(processor, enable_thinking=ENABLE_THINKING, dummy_answer=DUMMY_ASSISTANT_ANSWER),
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    return ds, loader

# optimizer steps
total_optimizer_steps = 0
for _, phase_df in PHASE_PLAN:
    phase_batches = math.ceil(len(phase_df) / PER_DEVICE_BATCH_SIZE)
    total_optimizer_steps += math.ceil(phase_batches / GRAD_ACCUM)

print("TOTAL_EPOCHS:", TOTAL_EPOCHS)
print("total_optimizer_steps:", total_optimizer_steps)

adamw_kwargs = dict(lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
try:
    optimizer = torch.optim.AdamW(
        (p for p in model.parameters() if p.requires_grad),
        fused=torch.cuda.is_available(),
        **adamw_kwargs,
    )
except TypeError:
    optimizer = torch.optim.AdamW(
        (p for p in model.parameters() if p.requires_grad),
        **adamw_kwargs,
    )

num_warmup_steps = max(1, int(total_optimizer_steps * WARMUP_RATIO))
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=max(total_optimizer_steps, 1),
)

scaler = torch.cuda.amp.GradScaler(enabled=(torch.cuda.is_available() and TORCH_DTYPE == torch.float16))

if loaded_for_resume:
    trainer_state_path = os.path.join(loaded_from, "trainer_state.pt")
    if os.path.exists(trainer_state_path):
        resume_state = torch.load(trainer_state_path, map_location="cpu")
        if resume_state.get("optimizer_state_dict") is not None:
            optimizer.load_state_dict(resume_state["optimizer_state_dict"])
        if resume_state.get("scheduler_state_dict") is not None:
            scheduler.load_state_dict(resume_state["scheduler_state_dict"])
        if resume_state.get("scaler_state_dict") is not None and scaler.is_enabled():
            scaler.load_state_dict(resume_state["scaler_state_dict"])

best_adapter_dir = os.path.join(RUN_DIR, "best_adapter")
final_adapter_dir = os.path.join(RUN_DIR, "final_adapter")

print("best_adapter_dir:", best_adapter_dir)
print("final_adapter_dir:", final_adapter_dir)


TOTAL_EPOCHS: 10
total_optimizer_steps: 5780


/tmp/ipykernel_14265/1494218713.py:62: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(torch.cuda.is_available() and TORCH_DTYPE == torch.float16))


best_adapter_dir: /content/drive/MyDrive/gemma4_31b_it_mcqa_lora_v6_human_bias/best_adapter
final_adapter_dir: /content/drive/MyDrive/gemma4_31b_it_mcqa_lora_v6_human_bias/final_adapter


# Fine-tuning

체크포인트 선정 기준을 바꿉니다.

1. **all_valid_acc** (사람식 라벨 분포를 더 직접 반영)
2. 동률이면 **clean_valid_acc**
3. 동률이면 **all_valid_ce** 감소
4. 마지막으로 **clean_valid_ce** 감소

즉, 이번 버전은 clean subset보다 **full valid의 실제 라벨 적합도**를 더 우선합니다.


In [ ]:

def save_checkpoint(
    epoch: int,
    model,
    processor,
    optimizer,
    scheduler,
    scaler,
    history: List[Dict[str, Any]],
    best_all_valid_acc: float,
    best_clean_valid_acc: float,
    best_all_valid_ce: float,
    best_clean_valid_ce: float,
    patience_counter: int,
):
    ckpt_dir = os.path.join(RUN_DIR, f"checkpoint-epoch-{epoch:02d}")
    os.makedirs(ckpt_dir, exist_ok=True)

    model.save_pretrained(ckpt_dir)
    processor.save_pretrained(ckpt_dir)

    trainer_state = {
        "epoch": epoch,
        "best_all_valid_acc": best_all_valid_acc,
        "best_clean_valid_acc": best_clean_valid_acc,
        "best_all_valid_ce": best_all_valid_ce,
        "best_clean_valid_ce": best_clean_valid_ce,
        "patience_counter": patience_counter,
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "history": history,
        "config": {
            "MODEL_ID": MODEL_ID,
            "PREV_RUN_DIR": PREV_RUN_DIR,
            "RUN_DIR": RUN_DIR,
            "INIT_FROM_CHECKPOINT": INIT_FROM_CHECKPOINT,
            "ENABLE_THINKING": ENABLE_THINKING,
            "VALID_RATIO": VALID_RATIO,
            "PER_DEVICE_BATCH_SIZE": PER_DEVICE_BATCH_SIZE,
            "GRAD_ACCUM": GRAD_ACCUM,
            "LEARNING_RATE": LEARNING_RATE,
            "WEIGHT_DECAY": WEIGHT_DECAY,
            "WARMUP_RATIO": WARMUP_RATIO,
            "MAX_GRAD_NORM": MAX_GRAD_NORM,
            "LORA_R": LORA_R,
            "LORA_ALPHA": LORA_ALPHA,
            "LORA_DROPOUT": LORA_DROPOUT,
            "USE_RSLORA": USE_RSLORA,
            "USE_LOFTQ_INIT": USE_LOFTQ_INIT,
            "ENABLE_VISION_TOWER_LORA": ENABLE_VISION_TOWER_LORA,
            "NUM_EPOCHS_STAGE1": NUM_EPOCHS_STAGE1,
            "NUM_EPOCHS_STAGE2": NUM_EPOCHS_STAGE2,
            "PROMPT_VARIANT_IDS_TRAIN": PROMPT_VARIANT_IDS_TRAIN,
            "PROMPT_VARIANT_IDS_EVAL": PROMPT_VARIANT_IDS_EVAL,
            "PROMPT_VARIANT_IDS_INFER": PROMPT_VARIANT_IDS_INFER,
            "TRAIN_OPTION_SHUFFLE_PROB": TRAIN_OPTION_SHUFFLE_PROB,
            "LABEL_SMOOTHING": LABEL_SMOOTHING,
            "HUMAN_TEACHER_WEIGHT": HUMAN_TEACHER_WEIGHT,
            "SEED": SEED,
        },
    }
    torch.save(trainer_state, os.path.join(ckpt_dir, "trainer_state.pt"))

    save_text(os.path.join(RUN_DIR, "latest_checkpoint.txt"), ckpt_dir)
    pd.DataFrame(history).to_csv(os.path.join(RUN_DIR, "train_history.csv"), index=False)
    return ckpt_dir

def is_better_checkpoint(
    all_valid_acc: float,
    clean_valid_acc: float,
    all_valid_ce: float,
    clean_valid_ce: float,
    best_all_valid_acc: float,
    best_clean_valid_acc: float,
    best_all_valid_ce: float,
    best_clean_valid_ce: float,
):
    if all_valid_acc > best_all_valid_acc + EARLY_STOP_MIN_DELTA:
        return True
    if abs(all_valid_acc - best_all_valid_acc) <= EARLY_STOP_MIN_DELTA:
        if clean_valid_acc > best_clean_valid_acc + EARLY_STOP_MIN_DELTA:
            return True
        if abs(clean_valid_acc - best_clean_valid_acc) <= EARLY_STOP_MIN_DELTA:
            if all_valid_ce < best_all_valid_ce - EARLY_STOP_MIN_DELTA:
                return True
            if abs(all_valid_ce - best_all_valid_ce) <= EARLY_STOP_MIN_DELTA:
                if clean_valid_ce < best_clean_valid_ce - EARLY_STOP_MIN_DELTA:
                    return True
    return False

for epoch in range(start_epoch, TOTAL_EPOCHS + 1):
    phase_name, phase_df = PHASE_PLAN[epoch - 1]
    print(f"\n========== Epoch {epoch}/{TOTAL_EPOCHS} | {phase_name} ==========")

    _, train_loader = build_train_loader(phase_df, epoch=epoch)

    model.train()
    optimizer.zero_grad(set_to_none=True)

    train_loss_sum = 0.0
    train_raw_acc_sum = 0.0
    train_weight_sum = 0.0
    train_batches = 0
    grad_accum_counter = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{TOTAL_EPOCHS} [train]", unit="batch")

    for step, batch in enumerate(train_bar, start=1):
        batch = move_batch_to_device(batch, target_device)

        with torch.autocast(
            device_type="cuda",
            dtype=TORCH_DTYPE,
            enabled=torch.cuda.is_available(),
        ):
            choice_logits = compute_choice_logits(model, batch, choice_token_ids)
            loss, per_sample_loss = compute_noise_robust_loss(
                choice_logits=choice_logits,
                target_probs=batch["target_probs"],
                sample_weights=batch["sample_weights"],
            )

        loss_to_backward = loss / GRAD_ACCUM

        if scaler.is_enabled():
            scaler.scale(loss_to_backward).backward()
        else:
            loss_to_backward.backward()

        with torch.no_grad():
            pred = choice_logits.argmax(dim=-1)
            correct = (pred == batch["gold_indices"]).float()
            weighted_correct = (correct * batch["sample_weights"]).sum().item()
            weight_sum = batch["sample_weights"].sum().item()

        train_loss_sum += float(loss.item())
        train_raw_acc_sum += weighted_correct
        train_weight_sum += weight_sum
        train_batches += 1
        grad_accum_counter += 1

        should_step = (grad_accum_counter == GRAD_ACCUM) or (step == len(train_loader))
        if should_step:
            if scaler.is_enabled():
                scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=MAX_GRAD_NORM,
            )

            if scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            grad_accum_counter = 0

        train_bar.set_postfix({
            "loss": f"{train_loss_sum / max(train_batches, 1):.4f}",
            "w_acc": f"{train_raw_acc_sum / max(train_weight_sum, 1e-8):.4f}",
        })

    avg_train_loss = train_loss_sum / max(train_batches, 1)
    avg_train_w_acc = train_raw_acc_sum / max(train_weight_sum, 1e-8)

    # -----------------------------
    # validation (human-like all_valid 우선)
    # -----------------------------
    model.eval()

    valid_all_logits = score_df_choice_logits(
        model,
        processor,
        valid_scored_df,
        prompt_variant_ids=PROMPT_VARIANT_IDS_EVAL,
        choice_token_ids=choice_token_ids,
        batch_size=1,
        desc=f"Epoch {epoch} valid_all",
        disable_adapter=False,
    )
    all_metrics = summarize_logits_against_df(valid_all_logits, valid_scored_df)

    clean_mask = valid_scored_df["eval_keep"].to_numpy().astype(bool)
    clean_valid_logits = valid_all_logits[clean_mask]
    clean_metrics = summarize_logits_against_df(clean_valid_logits, clean_valid_df)

    all_valid_acc = all_metrics["raw_acc"]
    clean_valid_acc = clean_metrics["raw_acc"]
    all_valid_ce = all_metrics["weighted_ce"]
    clean_valid_ce = clean_metrics["weighted_ce"]

    improved = is_better_checkpoint(
        all_valid_acc=all_valid_acc,
        clean_valid_acc=clean_valid_acc,
        all_valid_ce=all_valid_ce,
        clean_valid_ce=clean_valid_ce,
        best_all_valid_acc=best_all_valid_acc,
        best_clean_valid_acc=best_clean_valid_acc,
        best_all_valid_ce=best_all_valid_ce,
        best_clean_valid_ce=best_clean_valid_ce,
    )

    if improved:
        best_all_valid_acc = all_valid_acc
        best_clean_valid_acc = clean_valid_acc
        best_all_valid_ce = all_valid_ce
        best_clean_valid_ce = clean_valid_ce
        patience_counter = 0

        os.makedirs(best_adapter_dir, exist_ok=True)
        model.save_pretrained(best_adapter_dir)
        processor.save_pretrained(best_adapter_dir)
        save_text(os.path.join(RUN_DIR, "best_checkpoint.txt"), os.path.join(RUN_DIR, f"checkpoint-epoch-{epoch:02d}"))
    else:
        patience_counter += 1

    history.append({
        "epoch": epoch,
        "phase_name": phase_name,
        "train_loss": avg_train_loss,
        "train_weighted_acc": avg_train_w_acc,
        "all_valid_acc": all_valid_acc,
        "all_valid_weighted_ce": all_valid_ce,
        "clean_valid_acc": clean_valid_acc,
        "clean_valid_weighted_ce": clean_valid_ce,
        "best_all_valid_acc": best_all_valid_acc,
        "best_clean_valid_acc": best_clean_valid_acc,
        "best_all_valid_ce": best_all_valid_ce,
        "best_clean_valid_ce": best_clean_valid_ce,
        "patience_counter": patience_counter,
        "train_size": len(phase_df),
    })

    ckpt_dir = save_checkpoint(
        epoch=epoch,
        model=model,
        processor=processor,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        history=history,
        best_all_valid_acc=best_all_valid_acc,
        best_clean_valid_acc=best_clean_valid_acc,
        best_all_valid_ce=best_all_valid_ce,
        best_clean_valid_ce=best_clean_valid_ce,
        patience_counter=patience_counter,
    )

    print(
        f"[Epoch {epoch}] "
        f"phase={phase_name} | "
        f"train_loss={avg_train_loss:.4f} | "
        f"train_w_acc={avg_train_w_acc:.4f} | "
        f"all_valid_acc={all_valid_acc:.4f} | "
        f"clean_valid_acc={clean_valid_acc:.4f} | "
        f"all_valid_ce={all_valid_ce:.4f} | "
        f"clean_valid_ce={clean_valid_ce:.4f} | "
        f"checkpoint={ckpt_dir}"
    )

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}.")
        break

os.makedirs(final_adapter_dir, exist_ok=True)
model.save_pretrained(final_adapter_dir)
processor.save_pretrained(final_adapter_dir)

print("Best adapter:", best_adapter_dir)
print("Final adapter:", final_adapter_dir)


In [ ]:
try:
    del model
except Exception:
    pass
try:
    del base_model
except Exception:
    pass
gc.collect()

# Final inference

기본 동작:
- current v6 checkpoint들 + selected legacy checkpoint들 + optional base를 후보로 모음
- valid에서 다시 greedy search로 **최종 inference ensemble** 선택
- 선택된 조합으로 test inference 수행

즉, 이번 버전은 `best_adapter` 하나에 의존하지 않습니다.


In [ ]:
def build_inference_samples_for_row(
    row_like: Any,
    prompt_variant_ids: List[int],
    num_option_permutations: int = 1,
):
    row = row_to_dict(row_like)

    perms: List[Tuple[int, int, int, int]] = [(0, 1, 2, 3)]
    if num_option_permutations > 1:
        rng = random.Random(option_permutation_seed(row_id=row.get("id", 0), epoch=99991, salt=17))
        seen = {perms[0]}
        while len(perms) < num_option_permutations:
            perm = tuple(rng.sample([0, 1, 2, 3], 4))
            if perm not in seen:
                perms.append(perm)
                seen.add(perm)

    samples = []
    for prompt_variant in prompt_variant_ids:
        for perm in perms:
            row_cur, new_to_orig, _ = apply_option_permutation(row, list(perm))
            messages, img = build_messages_from_row(
                row_cur,
                prompt_variant=prompt_variant,
                assistant_answer=DUMMY_ASSISTANT_ANSWER,
            )
            samples.append({
                "messages": messages,
                "image": img,
                "gold_idx": 0,
                "target_probs": np.full(4, 0.25, dtype=np.float32),
                "sample_weight": 1.0,
                "row_id": row.get("id", 0),
                "new_to_orig": new_to_orig,
            })
    return samples


def score_inference_samples(model, processor, samples: List[Dict[str, Any]], choice_token_ids: torch.Tensor):
    collator = ChoiceLogitCollator(
        processor,
        enable_thinking=ENABLE_THINKING,
        dummy_answer=DUMMY_ASSISTANT_ANSWER,
    )

    logits_out = []
    for start in range(0, len(samples), INFER_VARIANT_BATCH_SIZE):
        chunk = samples[start : start + INFER_VARIANT_BATCH_SIZE]
        batch = collator(chunk)
        batch = move_batch_to_device(batch, target_device)

        with torch.no_grad():
            with torch.autocast(
                device_type="cuda",
                dtype=TORCH_DTYPE,
                enabled=torch.cuda.is_available(),
            ):
                logits = compute_choice_logits(model, batch, choice_token_ids)

        logits_out.extend([x.float().cpu() for x in logits])

    return logits_out

In [9]:

import glob

candidate_dirs = []
candidate_dirs.extend(sorted([p for p in glob.glob(os.path.join(RUN_DIR, "checkpoint-epoch-*")) if os.path.isdir(p)]))
candidate_dirs.extend([x["path"] for x in human_teacher_selection])
candidate_dirs = unique_keep_order([p for p in candidate_dirs if os.path.isdir(p)])

if not candidate_dirs:
    raise ValueError("No candidate adapters found for inference ensemble.")

print("[inference candidate adapters]")
for p in candidate_dirs:
    print(" -", p)

try:
    del model
except Exception:
    pass
try:
    del base_model
except Exception:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

infer_processor = AutoProcessor.from_pretrained(candidate_dirs[0])
choice_token_ids = build_choice_token_ids(infer_processor)

infer_base_model = load_quantized_base_model(MODEL_ID)
infer_model, infer_registry = build_multi_adapter_model(infer_base_model, candidate_dirs, prefix="infer")
infer_registry["base_model"] = {
    "path": MODEL_ID,
    "adapter_name": None,
    "kind": "base",
}

target_device = next(infer_model.parameters()).device
print("Inference target device:", target_device)

infer_clean_mask = valid_scored_df["eval_keep"].to_numpy().astype(bool)
infer_valid_logits_map = {}
infer_rows = []

for key, entry in infer_registry.items():
    cache_path = os.path.join(RUN_DIR, f"infer_valid_logits__{key}.pt")
    if os.path.exists(cache_path):
        logits = torch.load(cache_path, map_location="cpu").float()
        print("infer valid logits cache hit:", key)
    else:
        logits = score_registry_entry_logits(
            infer_model,
            infer_processor,
            valid_scored_df,
            entry,
            choice_token_ids=choice_token_ids,
            prompt_variant_ids=PROMPT_VARIANT_IDS_EVAL,
            desc=f"infer valid | {key}",
        ).float().cpu()
        torch.save(logits, cache_path)
        print("infer valid logits cache saved:", cache_path)

    infer_valid_logits_map[key] = logits
    metrics = evaluate_logits_bundle(logits, valid_scored_df, clean_mask=infer_clean_mask)
    infer_rows.append({
        "key": key,
        "path": entry["path"],
        "kind": entry["kind"],
        **metrics,
    })

infer_valid_table = pd.DataFrame(infer_rows).sort_values(
    ["all_valid_acc", "clean_valid_acc", "all_valid_ce"],
    ascending=[False, False, True],
).reset_index(drop=True)
print("\n[inference candidate ranking on valid]")
display(infer_valid_table)

infer_selected_keys, infer_weights, infer_valid_ensemble_logits, infer_search_trace = greedy_search_logits_map(
    infer_valid_logits_map,
    valid_scored_df,
    clean_mask=infer_clean_mask,
    max_models=MAX_INFER_ENSEMBLE_MODELS,
    alpha_grid=GREEDY_ALPHA_GRID,
)

print("\n[final inference ensemble selected]")
print("selected keys:", infer_selected_keys)
print("weights:", infer_weights)
display(infer_search_trace)

inference_ensemble_selection = [
    {
        "key": key,
        "path": infer_registry[key]["path"],
        "kind": infer_registry[key]["kind"],
        "weight": float(infer_weights[key]),
    }
    for key in infer_selected_keys
]
save_json(os.path.join(RUN_DIR, "final_inference_ensemble.json"), {
    "selected": inference_ensemble_selection,
    "weights": {k: float(v) for k, v in infer_weights.items()},
    "prompt_variant_ids": PROMPT_VARIANT_IDS_INFER,
    "num_option_permutations": INFER_NUM_OPTION_PERMUTATIONS,
})

preds = []
debug_rows = []

for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    pred = predict_row_with_registry_ensemble(
        infer_model,
        infer_processor,
        infer_registry,
        model_weights=infer_weights,
        row_like=row,
        choice_token_ids=choice_token_ids,
        prompt_variant_ids=PROMPT_VARIANT_IDS_INFER,
        num_option_permutations=INFER_NUM_OPTION_PERMUTATIONS,
    )
    preds.append(pred["pred_letter"])

    if i < 8:
        debug_rows.append({
            "id": row["id"],
            "pred": pred["pred_letter"],
            "p_a": float(pred["probs"][0]),
            "p_b": float(pred["probs"][1]),
            "p_c": float(pred["probs"][2]),
            "p_d": float(pred["probs"][3]),
        })

submission_path = os.path.join(RUN_DIR, "submission.csv")
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv(submission_path, index=False)

debug_path = os.path.join(RUN_DIR, "inference_debug_head.csv")
pd.DataFrame(debug_rows).to_csv(debug_path, index=False)

print("Saved submission:", submission_path)
print("Saved debug:", debug_path)
display(pd.DataFrame(debug_rows))


[inference candidate adapters]
 - /content/drive/MyDrive/gemma4_31b_it_mcqa_lora_v6_human_bias/checkpoint-epoch-01
 - /content/drive/MyDrive/gemma4_31b_it_mcqa_lora_v6_human_bias/checkpoint-epoch-02
 - /content/drive/MyDrive/gemma4_31b_it_mcqa_lora_v6_human_bias/checkpoint-epoch-03
 - /content/drive/MyDrive/gemma4_31b_it_mcqa_lora_v6_human_bias/checkpoint-epoch-04
 - /content/drive/MyDrive/gemma4_31b_it_mcqa_lora_v5/checkpoint-epoch-04
 - /content/drive/MyDrive/gemma4_31b_it_mcqa_lora_v5/checkpoint-epoch-02
 - /content/drive/MyDrive/gemma4_31b_it_mcqa_lora_v5/checkpoint-epoch-05
choice token a: [236746]
choice token b: [236763]
choice token c: [236755]
choice token d: [236753]


Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1310: UserWarning: You have requested `ensure_weight_tying`, but no tied modules are added in either `modules_to_save` or `target_modules`
  warnings.warn(


Inference target device: cuda:0
infer valid logits cache hit: infer_gemma4_31b_it_mcqa_lora_v6_human_bias_checkpoint_epoch_01
infer valid logits cache hit: infer_gemma4_31b_it_mcqa_lora_v6_human_bias_checkpoint_epoch_02
infer valid logits cache hit: infer_gemma4_31b_it_mcqa_lora_v6_human_bias_checkpoint_epoch_03


infer valid | infer_gemma4_31b_it_mcqa_lora_v6_human_bias_checkpoint_epoch_04 | prompt=0:   0%|          | 0/4…

...

,key,path,kind,all_valid_acc,all_valid_weighted_acc,all_valid_ce,clean_valid_acc,clean_valid_weighted_acc,clean_valid_ce
0,infer_gemma4_31b_it_mcqa_lora_v6_human_bias_ch...,/content/drive/MyDrive/gemma4_31b_it_mcqa_lora...,adapter,0.955556,0.977129,0.423741,0.955556,0.977129,0.423741
1,infer_gemma4_31b_it_mcqa_lora_v6_human_bias_ch...,/content/drive/MyDrive/gemma4_31b_it_mcqa_lora...,adapter,0.953086,0.975319,0.424413,0.953086,0.975319,0.424413
2,infer_gemma4_31b_it_mcqa_lora_v6_human_bias_ch...,/content/drive/MyDrive/gemma4_31b_it_mcqa_lora...,adapter,0.948148,0.970521,0.430347,0.948148,0.970521,0.430347
3,infer_gemma4_31b_it_mcqa_lora_v5_checkpoint_ep...,/content/drive/MyDrive/gemma4_31b_it_mcqa_lora...,adapter,0.948148,0.972903,0.500500,0.948148,0.972903,0.500500
4,infer_gemma4_31b_it_mcqa_lora_v5_checkpoint_ep...,/content/drive/MyDrive/gemma4_31b_it_mcqa_lora...,adapter,0.945679,0.971155,0.498537,0.945679,0.971155,0.498537
5,infer_gemma4_31b_it_mcqa_lora_v5_checkpoint_ep...,/content/drive/MyDrive/gemma4_31b_it_mcqa_lora...,adapter,0.943210,0.963366,0.560074,0.943210,0.963366,0.560074
6,infer_gemma4_31b_it_mcqa_lora_v6_human_bias_ch...,/content/drive/MyDrive/gemma4_31b_it_mcqa_lora...,adapter,0.933333,0.959045,0.446324,0.933333,0.959045,0.446324
7,base_model,google/gemma-4-31B-it,base,0.612346,0.663532,0.881039,0.612346,0.663532,0.881039



[final inference ensemble selected]
selected keys: ['infer_gemma4_31b_it_mcqa_lora_v6_human_bias_checkpoint_epoch_02', 'infer_gemma4_31b_it_mcqa_lora_v5_checkpoint_epoch_02', 'infer_gemma4_31b_it_mcqa_lora_v6_human_bias_checkpoint_epoch_04', 'infer_gemma4_31b_it_mcqa_lora_v6_human_bias_checkpoint_epoch_03']
weights: {'infer_gemma4_31b_it_mcqa_lora_v6_human_bias_checkpoint_epoch_02': 0.36125, 'infer_gemma4_31b_it_mcqa_lora_v5_checkpoint_epoch_02': 0.36125, 'infer_gemma4_31b_it_mcqa_lora_v6_human_bias_checkpoint_epoch_04': 0.12750000000000003, 'infer_gemma4_31b_it_mcqa_lora_v6_human_bias_checkpoint_epoch_03': 0.15000000000000002}


,round,selected,alpha,all_valid_acc,all_valid_weighted_acc,all_valid_ce,clean_valid_acc,clean_valid_weighted_acc,clean_valid_ce
0,0,infer_gemma4_31b_it_mcqa_lora_v6_human_bias_ch...,1.00,0.955556,0.977129,0.423741,0.955556,0.977129,0.423741
1,1,infer_gemma4_31b_it_mcqa_lora_v5_checkpoint_ep...,0.50,0.962963,0.981979,0.446539,0.962963,0.981979,0.446539
2,2,infer_gemma4_31b_it_mcqa_lora_v6_human_bias_ch...,0.15,0.962963,0.981788,0.439406,0.962963,0.981788,0.439406
3,3,infer_gemma4_31b_it_mcqa_lora_v6_human_bias_ch...,0.15,0.962963,0.981788,0.434701,0.962963,0.981788,0.434701


Inference:   0%|          | 0/5074 [00:00<?, ?sample/s]

NameError: name 'build_inference_samples_for_row' is not defined

In [ ]:

print("PREV_RUN_DIR:", PREV_RUN_DIR)
print("RUN_DIR:", RUN_DIR)
print("Init checkpoint:", INIT_FROM_CHECKPOINT)
print("Latest checkpoint:", read_text(os.path.join(RUN_DIR, "latest_checkpoint.txt")))
print("Best checkpoint:", read_text(os.path.join(RUN_DIR, "best_checkpoint.txt")))
print("Human teacher selection:", os.path.join(RUN_DIR, "human_teacher_selection.json"))
print("Final inference ensemble:", os.path.join(RUN_DIR, "final_inference_ensemble.json"))
print("Best adapter dir:", os.path.join(RUN_DIR, "best_adapter"))
print("Final adapter dir:", os.path.join(RUN_DIR, "final_adapter"))
print("Submission path:", os.path.join(RUN_DIR, "submission.csv"))
